# Removing the Seasonal Cycle: Daily Climatology vs. Harmonic Regression

---

## Overview

Before studying intraseasonal or interannual variability, we often remove the **seasonal cycle** to obtain **anomalies**, departures from the repeating annual pattern. This notebook compares two common approaches on daily **850-hPa zonal wind** ($u_{850}$) from the NCEP–NCAR Reanalysis and inspects the resulting spectra.

We will:

1. Load daily $u_{850}$ from the public Pythia Zarr store.
2. Remove the seasonal cycle with **daily climatology** and **harmonic regression**.
3. Compare the two anomaly fields at a grid point, over the tropical mean, and in the time-mean map.
4. Resample to monthly means, compute monthly anomalies, and apply the **FFT** to identify dominant periods.

An anomaly is defined as

$$
\text{anomaly} = \text{observed value} - \text{seasonal climatology}.
$$

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Remove the seasonal cycle using harmonic regression](./02.harmonic_anomalies.ipynb) | Necessary | Harmonic design matrix and least-squares fit |
| [The Discrete Fourier Transform](./03.discrete_fourier_transform.ipynb) | Helpful | Frequencies, periods, and Fourier coefficients |
| [Fourier Power Spectrum and Spectral Filtering](./04.fourier_spectral_analysis_tutorial.ipynb) | Helpful | Power spectrum and explained variance |

- **Time to learn**: 25 minutes

---

## Imports

All packages used in the rest of the notebook are imported up front. `s3fs` is configured to read anonymously from the public Pythia bucket on the Jetstream2 S3 endpoint.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import s3fs
import xarray as xr

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.2)

URL = "https://js2.jetstream-cloud.org:8001/"
fs = s3fs.S3FileSystem(anon=True, client_kwargs=dict(endpoint_url=URL))

## Load NCEP–NCAR zonal wind

We use daily **850-hPa zonal wind** from the NCEP–NCAR Reanalysis [@kanamitsu2002ncep, kalnay2018ncep], stored as a preprocessed Zarr dataset on the Pythia Jetstream2 bucket.

In [ ]:
uwind_ncep_ncar_store = s3fs.S3Map(
    root="pythia/uwind-ncep-ncar.zarr",
    s3=fs,
    check=False,
)

uwind_ncep_ncar = xr.open_zarr(uwind_ncep_ncar_store)

uwind_850_da = (
    uwind_ncep_ncar["uwnd"]
    .sel(level=850)
    .drop_vars("level", errors="ignore")
)

uwind_ncep_ncar

## Remove the seasonal cycle

We subtract the seasonal cycle at every grid point using two methods. **Daily climatology** averages all values for each calendar day (Jan 1, Jan 2, …) and subtracts that mean. **Harmonic regression** fits a smooth annual cycle as a sum of sine and cosine harmonics; see [Remove the seasonal cycle using harmonic regression](./02.harmonic_anomalies.ipynb) for the theory and design matrix.

In [ ]:
def remove_seasonal_cycle_daily_climatology(da, drop_feb29=True):
    """Remove the seasonal cycle using day-of-year climatology."""
    month_day = da["time"].dt.strftime("%m-%d")
    da = da.assign_coords(month_day=("time", month_day.data))

    if drop_feb29:
        da = da.where(da["month_day"] != "02-29", drop=True)

    climatology = da.groupby("month_day").mean("time")
    anomalies = da.groupby("month_day") - climatology
    return anomalies, climatology


def remove_seasonal_cycle_harmonic(data, n_harmonics=4, year_period=365.25):
    """Remove the seasonal cycle via harmonic least squares.

    Works on 3D arrays (time, lat, lon). Set ``year_period=12`` for monthly data.
    """
    n_time, n_lat, n_lon = data.shape
    data_2d = data.reshape(n_time, -1)

    t = np.arange(n_time)
    X = np.ones((n_time, 2 * n_harmonics + 1))
    for i in range(1, n_harmonics + 1):
        X[:, 2 * i - 1] = np.sin(i * 2 * np.pi * t / year_period)
        X[:, 2 * i] = np.cos(i * 2 * np.pi * t / year_period)

    coeffs = np.linalg.lstsq(X, data_2d, rcond=None)[0]
    return (data_2d - X @ coeffs).reshape(n_time, n_lat, n_lon)


def remove_seasonal_cycle_monthly_climatology(da):
    """Remove the seasonal cycle using monthly climatology (Jan–Dec means)."""
    climatology = da.groupby("time.month").mean("time")
    anomalies = da.groupby("time.month") - climatology
    return anomalies, climatology

In [ ]:
daily_anom, _ = remove_seasonal_cycle_daily_climatology(uwind_850_da)
daily_anom.attrs["long_name"] = "850 hPa zonal wind anomalies (daily climatology)"

harmonic_anom = uwind_850_da.copy(
    data=remove_seasonal_cycle_harmonic(uwind_850_da.values)
)
harmonic_anom.attrs["long_name"] = "850 hPa zonal wind anomalies (harmonic regression)"

daily_anom, harmonic_anom = xr.align(daily_anom, harmonic_anom, join="inner")

:::{note}
Harmonic regression with $N = 4$ pairs of sin/cos terms produces a smooth seasonal cycle and is less noisy than a raw daily climatology, especially for short records. We align the two anomaly fields because daily climatology may drop February 29.
:::

## Compare daily anomalies

We inspect both methods at one tropical grid point (0°N, 180°E), over the area-weighted tropical mean (30°S–30°N), and in the time-mean map to confirm that anomalies are near zero on average.

In [ ]:
lat_select = 0.0
lon_select = 180.0
time_idx = 10

daily_point = daily_anom.sel(lat=lat_select, lon=lon_select, method="nearest")
harmonic_point = harmonic_anom.sel(
    lat=lat_select, lon=lon_select, method="nearest")
difference = daily_anom - harmonic_anom
difference_point = difference.sel(
    lat=lat_select, lon=lon_select, method="nearest")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
daily_anom.isel(time=time_idx).plot(ax=axes[0])
axes[0].set_title(
    f"Anomaly from daily climatology ({pd.to_datetime(daily_anom.time[time_idx].values):%Y-%m-%d})")
harmonic_anom.isel(time=time_idx).plot(ax=axes[1])
axes[1].set_title(
    f"Anomaly from harmonic regression ({pd.to_datetime(harmonic_anom.time[time_idx].values):%Y-%m-%d})")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

daily_point.plot(ax=axes[0], label="Daily climatology", alpha=0.8)
harmonic_point.plot(ax=axes[0], label="Harmonic regression", alpha=0.8)
axes[0].axhline(0, color="k", linewidth=0.8)
axes[0].set_title(
    f"850 hPa zonal-wind anomalies at lat={lat_select}, lon={lon_select}")
axes[0].set_ylabel("u-wind anomaly (m/s)")
axes[0].legend()
axes[0].set_xlim(pd.Timestamp("1995-01-01"), pd.Timestamp("1999-01-01"))

difference_point.plot(ax=axes[1], color="green")
axes[1].set_title(
    f"Difference between daily and harmonic anomalies at lat={lat_select}, lon={lon_select}")
axes[1].axhline(0, color="k", linewidth=0.8)
axes[1].set_ylabel("Difference (m/s)")
axes[1].set_xlabel("Year")
axes[1].set_xlim(pd.Timestamp("1995-01-01"), pd.Timestamp("1999-01-01"))
plt.tight_layout()
plt.show()

In [ ]:
weights = np.cos(np.deg2rad(daily_anom["lat"]))
daily_tropical_mean = daily_anom.weighted(weights).mean(("lat", "lon"))
harmonic_tropical_mean = harmonic_anom.weighted(weights).mean(("lat", "lon"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

daily_tropical_mean.plot(
    ax=axes[0], label="Anomaly from daily climatology", alpha=0.8)
harmonic_tropical_mean.plot(
    ax=axes[0], label="Anomaly from harmonic regression", alpha=0.8)
axes[0].axhline(0, color="k", linewidth=0.8)
axes[0].set_title("Tropical-mean anomalies")
axes[0].set_ylabel("u-wind anomaly (m/s)")
axes[0].legend()

daily_anom.mean("time").plot(ax=axes[1], cmap="RdBu_r", add_colorbar=True)
axes[1].set_title("Time mean of daily-climatology anomalies (m/s)")
plt.tight_layout()
plt.show()

The time-mean anomaly map should be close to zero everywhere; values close to zero confirm that the seasonal cycle has been removed.

## Fourier analysis of monthly anomalies

For spectral analysis we resample to **monthly means**, which suppresses day-to-day noise and makes the annual harmonics easier to interpret. At the selected grid point we remove the seasonal cycle again (monthly climatology vs. harmonic regression with `year_period=12`), then apply the FFT to see which periods explain the most variance.

In [ ]:
uwind_850_monthly = uwind_850_da.resample(time="MS").mean()
uwind_850_series = uwind_850_monthly.sel(
    lat=lat_select, lon=lon_select, method="nearest"
)

monthly_clim_anom, _ = remove_seasonal_cycle_monthly_climatology(uwind_850_monthly)
monthly_harm_anom = xr.DataArray(
    remove_seasonal_cycle_harmonic(
        uwind_850_monthly.values, n_harmonics=4, year_period=12
    ),
    coords=uwind_850_monthly.coords,
    dims=uwind_850_monthly.dims,
)

monthly_clim_point = monthly_clim_anom.sel(
    lat=lat_select, lon=lon_select, method="nearest"
)
monthly_harm_point = monthly_harm_anom.sel(
    lat=lat_select, lon=lon_select, method="nearest"
)
monthly_clim_at_point = uwind_850_series.groupby("time.month").mean("time")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
uwind_850_series.plot(ax=axes[0])
axes[0].set_title(f"Monthly mean $u_{{850}}$ at lat={lat_select}, lon={lon_select}")
monthly_clim_at_point.plot(ax=axes[1], marker="o")
axes[1].set_title("Monthly climatology at selected point")
axes[1].set_xticks(np.arange(1, 13))
plt.tight_layout()
plt.show()

In [ ]:
def fft_percent_variance(series, sampling_interval=1):
    """Return positive periods (months) and percent variance explained."""
    centered = series - np.mean(series)
    n = len(centered)
    freqs = np.fft.fftfreq(n, d=sampling_interval)
    coeffs = np.fft.fft(centered)
    power = np.abs(coeffs) ** 2
    pct = (power / np.sum(power)) * 100.0

    periods = 1 / freqs
    mask = np.isfinite(periods) & (periods > 0) & (pct > 0)
    sort_idx = np.argsort(periods[mask])
    return periods[mask][sort_idx], pct[mask][sort_idx]


reference_periods = [12, 6, 4, 3]  # annual, semi-annual, and higher harmonics

clim_periods, clim_pct = fft_percent_variance(monthly_clim_point.values)
harm_periods, harm_pct = fft_percent_variance(monthly_harm_point.values)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(clim_periods, clim_pct, label="Monthly climatology anomalies")
ax.semilogx(harm_periods, harm_pct, label="Harmonic regression anomalies", alpha=0.8)
for period in reference_periods:
    ax.axvline(period, color="gray", linestyle="--", linewidth=0.8, zorder=0)
ax.set_xlabel("Period (months)")
ax.set_ylabel("Percent variance explained (%)")
ax.set_title(f"Fourier spectrum at lat={lat_select}, lon={lon_select}")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

Vertical dashed lines mark the annual cycle (12 months) and its harmonics. After removing the seasonal cycle, power at those periods should be small; remaining peaks reflect intraseasonal, interannual, and longer-term variability.

---

## Summary

We computed **850-hPa zonal-wind anomalies** by subtracting either a daily (or monthly) climatology or a harmonic fit of the seasonal cycle. The two daily methods agree closely at most locations; differences are largest where the raw daily climatology is noisier. Fourier analysis of monthly anomalies confirms that seasonal power is suppressed and highlights variability on other time scales.

### What's next?

The next notebook, [Frequency Decomposition of Zonal Wind Variability Using the Fourier Transform](./09.freq-decomp.ipynb), extends this idea: it partitions the total variance of $u_{850}$ into annual, intraseasonal, interannual, background, and subseasonal bands and maps the spatial structure of each.